# Global SST and U-Net: one fixed combination

This notebook evaluates one candidate using **saved prediction maps** for January 2021–December 2022:

```text
Candidate = 0.75 × hybrid with 8 SST PCs + 0.25 × existing U-Net
Reference = 0.75 × original hybrid       + 0.25 × the same U-Net
```

It tests whether the SST contribution persists in the existing blend. Comparing the candidate with the SST hybrid alone also shows whether the U-Net adds anything after SST is present. There are **zero model fits**, no search over weights and no source-provider downloads. Use CPU, with the accelerator disabled. Runtime is mostly file checks, NetCDF I/O and scoring, not training.

These years have already been evaluated in both source experiments. This is a retrospective development comparison, **not an independent holdout**. Source models were fitted through September 2020 with the existing lag policy; historical publication vintages remain unverified. No operational model is replaced automatically.

## Inputs: observations and two saved prediction files

Only these three files are needed. You do not need to attach raw SST, SEAS5 or CFSv2 for this test.

| Input | Where to find it | SHA-256 prefix |
|---|---|---|
| `treino_tp.nc` | Original competition input; use the unshifted rainfall file | `012bebbe0e38` |
| SST `predictions.nc` | Complete output of the SST extension: `sst_temporal_extension/results/models/predictions.nc` | `8888ce1301a3` |
| U-Net `predictions.nc` | Complete output of the earlier hybrid/U-Net extension, Kaggle version **354026629** | `5e5341220a58` |

The U-Net run is [available here](https://www.kaggle.com/code/houxie/worcap-hybrid-and-u-net-temporal-extension/output?scriptVersionId=354026629). Use its complete output, not `temporal_extension_reports.zip`. Likewise, `sst_temporal_extension_reports.zip` does **not** contain the SST maps.

Import this notebook into a new Kaggle notebook. Add the competition data and the two saved notebook outputs through **Add Input → Notebook**, selecting the appropriate saved versions. Alternatively, download each `predictions.nc` and upload them in separate folders within a private dataset. Both files share the same basename; keep them in different folders. If renamed, set their complete file paths in the configuration cell.

Automatic discovery searches `/kaggle/input` and `/kaggle/working` and checks the **full hashes**, not just filenames. Identical copies are harmless; a missing or different run stops the notebook before calculation. A sidebar entry containing only notebook HTML/code is not sufficient.

## Checks and computation

1. Verify both source map hashes and the original observation hash against the archived receipts.
2. Require all 24 months, the same 301 × 261 grid, correct units and finite nonnegative rainfall. Reject reordered or mismatched coordinates.
3. Require identical hybrid and climatology maps across both experiments. Reconstruct the archived reference blend exactly.
4. Form the candidate in float64 from the already reconstructed rainfall maps, with no additional clipping or correction. Save it and check an exact readback before opening evaluation labels.
5. Reproduce the archived hybrid, SST-hybrid, U-Net, climatology and reference-blend scores. Then compare the new candidate globally, annually, monthly and by latitude band.
6. Check both equivalent fixed-blend MSE identities using the paired error products. Error correlation is descriptive; no optimized weight is computed.

The report includes RMSE, MAE, signed/absolute bias differences and area-weighted RMSE. A lower aggregate RMSE alone does not imply that every year, month or region improved. Preserve the full output for later diagnostics.

## Run and return

Select **Run All**. The synthetic tests in the first section check implementation behavior and do not measure forecast skill. The next section locates the three inputs; the final section evaluates the real saved maps.

Completion is reported as `complete_comparison: true`. `complete_seven_blocks: false` is expected because this notebook evaluates one 24-month extension. Download **`sst_unet_blend_reports.zip`**. It contains the small reports, code snapshot and checks; full candidate maps remain in the saved output.

If a completed run is found with the same inputs/code/environment, the notebook verifies and reuses it. An interrupted output is preserved; select a fresh output directory before retrying.


In [ ]:
from pathlib import Path
import hashlib, json, sys, runpy, shutil
BASE = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'outputs'
RUN_ROOT = BASE/'sst_unet_fixed_blend'
CODE_ROOT = RUN_ROOT/'code'
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = {'research/common/inputs.py': '"""Load the existing audited inputs; no downloads, imputation or new sources."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nLIBRARY = ROOT / \'research/lagged_sources/library.py\'\nREQUIRED = {\'numpy\': \'2.0.2\', \'pandas\': \'2.3.3\', \'xarray\': \'2025.12.0\', \'lightgbm\': \'4.6.0\'}\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef write_json(path, obj):\n    Path(path).write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False, default=str) + \'\\n\', encoding=\'utf-8\')\n\n\ndef library(path=LIBRARY):\n    # Separate module globals: callers cannot change another run\'s index table.\n    spec = importlib.util.spec_from_file_location(\'fixed_lag_library\', path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef find_unique(filename, supplied=None):\n    if supplied:\n        p = Path(supplied)\n        require((p / filename).is_file(), f\'Missing {filename} in {p}\')\n        return p.resolve()\n    candidates = sorted({p.parent.resolve() for p in Path(\'/kaggle/input\').rglob(filename)})\n    require(len(candidates) == 1, f\'Attach exactly one input containing {filename}, or set its directory explicitly. Found {len(candidates)}.\')\n    return candidates[0]\n\n\ndef preflight(official=None, seas5=None, cfsv2=None, final=False):\n    paths = dict(official=find_unique(\'treino_tp.nc\', official),\n                 seas5=find_unique(\'seas5_51_manifesto.json\', seas5),\n                 cfsv2=find_unique(\'cfsv2_manifesto.json\', cfsv2))\n    expected = json.loads((ROOT / \'research/lagged_sources/official_hashes.json\').read_text())\n    files = {}\n    for row in expected:\n        if not row[\'nome\'].startswith(\'treino_\'):\n            continue\n        p = paths[\'official\'] / row[\'nome\']\n        require(p.is_file(), f\'Missing official input: {p.name}\')\n        require(sha256(p) == row[\'sha256\'], f\'Official input hash mismatch: {p.name}\')\n        files[p.name] = row[\'sha256\']\n    for source, name in [(\'seas5\', \'seas5_51_manifesto.json\'), (\'cfsv2\', \'cfsv2_manifesto.json\')]:\n        manifest = paths[source] / name\n        files[name] = sha256(manifest)\n        m = json.loads(manifest.read_text(encoding=\'utf-8\'))\n        for phase in [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else []):\n            row = m[\'arquivos\'][phase]\n            p = paths[source] / row[\'arquivo\']\n            require(p.is_file() and sha256(p) == row[\'sha256\'], f\'Missing or altered {source}: {phase}\')\n            files[p.name] = row[\'sha256\']\n    versions = {n: importlib.metadata.version(n) for n in REQUIRED}\n    require(versions == REQUIRED, f\'The fixed baseline needs {REQUIRED}; found {versions}.\')\n    return paths, dict(files=files, versions=versions, final_training=final)\n\n\nclass Inputs:\n    def __init__(self, official=None, seas5=None, cfsv2=None, output=\'outputs\', final=False):\n        self.paths, self.audit = preflight(official, seas5, cfsv2, final)\n        self.lib = m = library()\n        m.PASTA = self.paths[\'official\']\n        m.PASTA_SEAS5_MANUAL = self.paths[\'seas5\']\n        m.PASTA_CFSV2_MANUAL = self.paths[\'cfsv2\']\n        m.PASTA_SEAS5, m.MANIFESTO_SEAS5 = m.localizar_seas5()\n        m.PASTA_CFSV2, m.MANIFESTO_CFSV2 = m.localizar_cfsv2()\n        m.SAIDA = Path(output)\n        m.SAIDA.mkdir(parents=True, exist_ok=True)\n        end = \'2022-12-01\' if final else \'2020-12-01\'\n        start = \'1993-01-01\' if final else \'1976-10-01\'\n        with xr.open_dataset(m.PASTA / \'treino_tp.nc\') as ds:\n            require(ds.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected rainfall units.\')\n            self.rain = ds.tp.sel(time=slice(start, end)).transpose(\'time\', \'lat\', \'lon\').load()\n        require(pd.DatetimeIndex(self.rain.time.values).equals(pd.date_range(start, end, freq=\'MS\')), \'Incomplete rainfall calendar.\')\n        require(np.array_equal(self.rain.lat, np.arange(-60, 15.25, .25)) and\n                np.array_equal(self.rain.lon, np.arange(-90, -24.75, .25)), \'Different target grid.\')\n        require(np.isfinite(self.rain.values).all() and (self.rain.values >= 0).all(), \'Invalid rainfall.\')\n        atm_start = pd.Timestamp(start) - pd.DateOffset(months=4)\n        atm_end = pd.Timestamp(end) - pd.DateOffset(months=4)\n        self.atmosphere, _ = m.carregar_atmosfera(self.rain, atm_start, atm_end)\n        index_path = ROOT / (\'competition/metadata/NOAA/indices_noaa.csv\' if final else \'research/lagged_sources/ocean_indices.csv\')\n        m.INDICES_OC = pd.read_csv(index_path, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n        m.conferir_indices(m.INDICES_OC)\n        self.audit[\'indices_sha256\'] = sha256(index_path)\n        self.audit[\'library_sha256\'] = sha256(LIBRARY)\n        phases = [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else [])\n        self.seas = xr.concat([m.carregar_seas5(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.cfs = xr.concat([m.carregar_cfsv2(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.audit.update(rainfall_read=[start, end], forecast_partitions=phases,\n                          test_partition_read=False, sst_used=False)\n\n\nclass Context:\n    """Training-only climatologies, with a bounded reference window for inner selection."""\n    def __init__(self, data, reference, training):\n        self.data = data\n        m = data.lib\n        self.reference = pd.DatetimeIndex(reference)\n        self.training = pd.DatetimeIndex(training)\n        require(24 <= len(self.training) <= 360 and len(self.reference) <= 360, \'Invalid training window.\')\n        require(self.training.isin(self.reference).all() and self.reference.is_monotonic_increasing and\n                self.reference.is_unique and self.training.is_unique, \'Invalid training/reference dates.\')\n        self.cutoff = self.reference[-1]\n        rain = data.rain.sel(time=self.reference).values\n        self.counts = np.array([(self.reference.month == month).sum() for month in range(1, 13)])\n        require((self.counts > 1).all(), \'At least two reference years per month are needed.\')\n        self.sums = np.stack([rain[self.reference.month == month].sum(0, dtype=np.float64) for month in range(1, 13)])\n        self.climate = xr.DataArray((self.sums / self.counts[:, None, None]).astype(\'float32\'),\n            dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=data.rain.lat, lon=data.rain.lon), attrs=dict(units=\'mm/day\'))\n        origins = m.origem_mensal(self.reference, 4)\n        self.atm = m.ajustar_clima_atmosfera(data.atmosphere, origins)\n        origins = m.origem_mensal(self.reference, 3)\n        vals = m.INDICES_OC.loc[origins, m.INDICES_NOMES].to_numpy(dtype=np.float64)\n        self.atm[\'_clima_indices\'] = np.stack([vals[origins.month == month].mean(0) for month in range(1, 13)])\n        self.seasonal = []\n        for forecasts in [data.seas, data.cfs]:\n            v = forecasts.sel(time=self.training)\n            require(pd.DatetimeIndex(v.time_origem.values).equals(m.origem_mensal(self.training, 1)), \'Wrong seasonal initialization.\')\n            means = np.stack([v.values[self.training.month == month].mean(0, dtype=np.float64) for month in range(1, 13)]).astype(\'float32\')\n            require(np.isfinite(means).all(), \'Missing seasonal reference month.\')\n            self.seasonal.append(xr.DataArray(means, dims=self.climate.dims, coords=self.climate.coords))\n\n    def features(self, target, training=False):\n        d, m = self.data, self.data.lib\n        t = pd.Timestamp(target)\n        require(t in self.training if training else t > self.cutoff, \'Month crosses the training boundary.\')\n        x = np.column_stack([m.matriz_mes(d.atmosphere, self.atm, self.climate, t),\n            m.valores_seas5(d.seas, t), m.valores_anomalia_seas5(d.seas, self.seasonal[0], t),\n            m.valores_cfsv2(d.cfs, t), m.valores_anomalia_cfsv2(d.cfs, self.seasonal[1], t)]).astype(\'float32\')\n        if training:\n            # Exclude the entire target map, including neighbors seen by convolutions.\n            y = d.rain.sel(time=t).values\n            month = t.month - 1\n            x[:, 22] = ((self.sums[month] - y.astype(\'float64\')) / (self.counts[month] - 1)).ravel().astype(\'float32\')\n        require(np.isfinite(x).all(), \'Nonfinite features.\')\n        return x.T.reshape(31, d.rain.sizes[\'lat\'], d.rain.sizes[\'lon\'])\n\n    def save(self, folder):\n        folder = Path(folder)\n        folder.mkdir(parents=True, exist_ok=True)\n        self.climate.to_netcdf(folder / \'rainfall_climatology.nc\')\n        np.savez_compressed(folder / \'references.npz\', sums=self.sums, counts=self.counts,\n                            seas5=self.seasonal[0].values, cfsv2=self.seasonal[1].values, **self.atm)\n        write_json(folder / \'context.json\', dict(reference=self.reference.strftime(\'%Y-%m\').tolist(),\n                                                training=self.training.strftime(\'%Y-%m\').tolist(), features=self.data.lib.FEATURES_MULTISSISTEMA))\n', 'research/diagnostics/analyze_errors.py': '"""Audit saved maps and describe errors without fitting or selecting a model.\n\nRun from the repository root with --predictions, --observations and --output.\nThe observations must be the original treino_tp.nc, not the shifted tp_alvo.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\n\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nROOT = Path(__file__).resolve().parents[2]\nEVIDENCE = ROOT / \'research/spatial_unet/evidence/seven_blocks\'\nBLOCKS = (\'H1\', \'H2\', \'H3\', \'H4\', \'A\', \'B\', \'C\')\nMODELS = (\'hybrid\', \'unet\', \'climatology\')\nBINS = np.array([0., 1., 3., 5., 10., np.inf])\nBIN_NAMES = (\'[0,1)\', \'[1,3)\', \'[3,5)\', \'[5,10)\', \'[10,infinity)\')\nCOLORS = dict(hybrid=\'#087e8b\', unet=\'#db654a\', climatology=\'#9299a1\')\nSUMS = [\'n\', \'sse\', \'error_sum\', \'absolute_error_sum\', \'observed_sum\', \'predicted_sum\']\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef save_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\', newline=\'\\n\')\n\n\ndef stats(observed, predicted):\n    observed, predicted = np.asarray(observed, dtype=\'float64\'), np.asarray(predicted, dtype=\'float64\')\n    require(observed.shape == predicted.shape, \'Different observation/prediction shapes.\')\n    require(np.isfinite(observed).all() and np.isfinite(predicted).all(), \'Nonfinite rainfall.\')\n    error = predicted - observed\n    return dict(n=error.size, sse=float(np.square(error).sum()), error_sum=float(error.sum()),\n                absolute_error_sum=float(np.abs(error).sum()), observed_sum=float(observed.sum()),\n                predicted_sum=float(predicted.sum()))\n\n\ndef pooled(frame, groups):\n    result = frame.groupby(groups, as_index=False, sort=False, observed=True)[SUMS].sum()\n    denominator = result.n.replace(0, np.nan)\n    result[\'rmse\'] = np.sqrt(result.sse / denominator)\n    result[\'mae\'] = result.absolute_error_sum / denominator\n    result[\'bias\'] = result.error_sum / denominator\n    result[\'observed_mean\'] = result.observed_sum / denominator\n    result[\'predicted_mean\'] = result.predicted_sum / denominator\n    return result\n\n\ndef intensity_masks(observed):\n    require(np.isfinite(observed).all() and (observed >= 0).all(), \'Invalid observed rainfall.\')\n    return [(observed >= low) & (observed < high) for low, high in zip(BINS[:-1], BINS[1:])]\n\n\ndef exact_alignment(observed, forecast, dates):\n    require(set(forecast.data_vars) == set(MODELS), \'Unexpected forecast variables.\')\n    require(pd.DatetimeIndex(forecast.time.values).equals(dates), \'Wrong or reordered forecast dates.\')\n    for name in MODELS:\n        require(forecast[name].dims == (\'time\', \'lat\', \'lon\'), \'Wrong forecast dimensions.\')\n        require(forecast[name].attrs.get(\'units\') == \'mm/day\', \'Wrong forecast units.\')\n    # Exact alignment rejects flipped, reordered or merely similar coordinates.\n    obs = observed.sel(time=dates).transpose(\'time\', \'lat\', \'lon\')\n    obs, forecast = xr.align(obs, forecast, join=\'exact\')\n    require(np.isfinite(obs.values).all() and (obs.values >= 0).all(), \'Invalid observations.\')\n    for name in MODELS:\n        require(np.isfinite(forecast[name]).all() and (forecast[name] >= 0).all(), \'Invalid predictions.\')\n    return obs, forecast\n\n\ndef verify_file(path, expected):\n    actual = digest(path)\n    require(actual == expected, f\'Hash mismatch: {Path(path).name}\')\n    return actual\n\n\ndef comparison(table, keys, total_n):\n    hybrid = table[table.model == \'hybrid\'].set_index(keys)\n    unet = table[table.model == \'unet\'].set_index(keys).reindex(hybrid.index)\n    require(np.array_equal(hybrid.n, unet.n), \'Unpaired diagnostic groups.\')\n    result = hybrid[[\'n\', \'rmse\', \'mae\', \'bias\']].rename(columns={c: \'hybrid_\' + c for c in [\'rmse\', \'mae\', \'bias\']})\n    for c in [\'rmse\', \'mae\', \'bias\']:\n        result[\'unet_\' + c] = unet[c]\n        result[\'delta_\' + c] = unet[c] - hybrid[c]\n    result[\'delta_sse\'] = unet.sse - hybrid.sse\n    result[\'contribution_to_global_delta_mse\'] = result.delta_sse / total_n\n    return result.reset_index()\n\n\ndef figure_setup():\n    plt.rcParams.update({\'font.size\': 10, \'axes.spines.top\': False, \'axes.spines.right\': False,\n                         \'figure.facecolor\': \'white\', \'savefig.facecolor\': \'white\'})\n\n\ndef plot_results(tables, maps, histories, output):\n    figure_setup()\n    monthly = tables[\'calendar_comparison\']\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)\n    for model in MODELS:\n        data = tables[\'calendar_month\'].query(\'model == @model\').sort_values(\'calendar_month\')\n        axes[0].plot(data.calendar_month, data.rmse, marker=\'o\', label=model.title(), color=COLORS[model])\n    axes[0].set(xlabel=\'Target calendar month\', ylabel=\'Pooled RMSE (mm/day)\', xticks=range(1, 13))\n    axes[0].legend(frameon=False)\n    axes[1].bar(monthly.calendar_month, monthly.delta_rmse,\n                color=np.where(monthly.delta_rmse > 0, COLORS[\'unet\'], COLORS[\'hybrid\']))\n    axes[1].axhline(0, color=\'#555\', linewidth=.8)\n    axes[1].set(xlabel=\'Target calendar month\', ylabel=\'U-Net minus hybrid RMSE (mm/day)\', xticks=range(1, 13))\n    fig.suptitle(\'Calendar-month errors | 2007–2020 development years\')\n    fig.savefig(output / \'calendar_month.png\', dpi=155); plt.close(fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)\n    for model in MODELS:\n        data = tables[\'intensity\'].query(\'model == @model\').set_index(\'intensity\').loc[list(BIN_NAMES)]\n        axes[0].plot(range(5), data.rmse, marker=\'o\', label=model.title(), color=COLORS[model])\n        axes[1].plot(range(5), data.bias, marker=\'o\', color=COLORS[model])\n    for ax in axes:\n        ax.set_xticks(range(5), [\'0–<1\', \'1–<3\', \'3–<5\', \'5–<10\', \'≥10\'])\n        ax.set_xlabel(\'Observed monthly mean rainfall (mm/day)\')\n    axes[0].set_ylabel(\'Conditional RMSE (mm/day)\'); axes[0].legend(frameon=False)\n    axes[1].set_ylabel(\'Conditional bias: predicted − observed (mm/day)\')\n    axes[1].axhline(0, color=\'#555\', linewidth=.8)\n    fig.suptitle(\'Rainfall intensity | fixed descriptive bins, not daily extremes\')\n    fig.savefig(output / \'rainfall_intensity.png\', dpi=155); plt.close(fig)\n\n    fig, axes = plt.subplots(1, 3, figsize=(12, 6.7), constrained_layout=True)\n    delta = maps.unet_rmse - maps.hybrid_rmse\n    vmax = float(max(maps.hybrid_rmse.max(), maps.unet_rmse.max()))\n    for ax, name in zip(axes[:2], [\'hybrid\', \'unet\']):\n        artist = ax.pcolormesh(maps.lon, maps.lat, maps[name + \'_rmse\'], shading=\'auto\', cmap=\'viridis\', vmin=0, vmax=vmax)\n        ax.set_title(name.title() + \' RMSE\')\n    fig.colorbar(artist, ax=list(axes[:2]), label=\'RMSE (mm/day)\', orientation=\'horizontal\', shrink=.9)\n    bound = float(np.abs(delta).max())\n    artist = axes[2].pcolormesh(maps.lon, maps.lat, delta, shading=\'auto\', cmap=\'RdBu_r\', vmin=-bound, vmax=bound)\n    axes[2].set_title(\'U-Net − hybrid RMSE\')\n    fig.colorbar(artist, ax=axes[2], label=\'Difference (mm/day)\', orientation=\'horizontal\')\n    for ax in axes:\n        ax.set(xlabel=\'Longitude (°)\', ylabel=\'Latitude (°)\', xlim=(-90, -25), ylim=(-60, 15), aspect=\'equal\')\n    fig.suptitle(\'Errors at each grid cell | all 168 months, full supplied domain\')\n    fig.savefig(output / \'spatial_errors.png\', dpi=155); plt.close(fig)\n\n    fig, axes = plt.subplots(4, 2, figsize=(12, 12), constrained_layout=True)\n    for ax, (block, inner, outer, selected) in zip(axes.flat, histories):\n        ax.plot(inner.epoch, np.sqrt(inner.training_mse), label=\'Inner training √MSE (online)\', color=\'#9299a1\')\n        ax.plot(inner.epoch, inner.inner_rmse, label=\'Inner validation RMSE\', color=COLORS[\'unet\'])\n        ax.plot(outer.epoch, np.sqrt(outer.training_mse), \'--\', label=\'Outer training √MSE (online)\', color=COLORS[\'hybrid\'])\n        ax.axvline(selected, color=\'#333\', linestyle=\':\', label=\'Selected epoch\')\n        ax.set(title=f\'{block} | selected epoch {selected}\', xlabel=\'Epoch\', ylabel=\'mm/day\')\n    axes.flat[-1].axis(\'off\')\n    handles, labels = axes.flat[0].get_legend_handles_labels()\n    axes.flat[-1].legend(handles, labels, loc=\'upper left\', frameon=False)\n    axes.flat[-1].text(.05, .12, \'Training losses use evolving weights and unclipped\\nanomalies. Validation uses a fixed end-of-epoch\\nmodel and clipped rainfall; gaps are not directly\\ncomparable generalization-error estimates.\', transform=axes.flat[-1].transAxes, fontsize=9)\n    fig.suptitle(\'Training histories | original epoch selection, no new fitting\')\n    fig.savefig(output / \'training_curves.png\', dpi=150); plt.close(fig)\n\n\ndef execute(predictions, observations, output, evidence=EVIDENCE):\n    predictions, observations, output, evidence = map(Path, (predictions, observations, output, evidence))\n    require(not output.exists(), \'Choose a new output directory; existing reports are preserved.\')\n    signature = read_json(evidence / \'signature.json\')\n    signature_hash = digest(evidence / \'signature.json\')\n    inputs = {\'observations\': verify_file(observations, signature[\'inputs\'][\'files\'][\'treino_tp.nc\']), \'signature\': signature_hash}\n    monthly_rows, intensity_rows, histories, training_rows = [], [], [], []\n    pixel_sums = None\n    for block_index, block in enumerate(BLOCKS):\n        completed = read_json(evidence / block / \'complete.json\')\n        require(completed[\'signature\'] == signature_hash, \'Run signature mismatch.\')\n        p = predictions / block / \'predictions.nc\'\n        inputs[block + \'/predictions.nc\'] = verify_file(p, completed[\'files\'][\'predictions.nc\'])\n        for name in [\'monthly_metrics.csv\', \'inner/training_history.json\', \'inner/training.json\', \'unet/training_history.json\', \'unet/training.json\']:\n            inputs[block + \'/\' + name] = verify_file(evidence / block / name, completed[\'files\'][name])\n        dates = pd.date_range(f\'{2007 + 2 * block_index}-01-01\', periods=24, freq=\'MS\')\n        archived = pd.read_csv(evidence / block / \'monthly_metrics.csv\')\n        with xr.open_dataset(observations) as source, xr.open_dataset(p) as ds:\n            require(source.tp.attrs.get(\'units\') == \'mm/day\', \'Wrong observation units.\')\n            observed, ds = exact_alignment(source.tp, ds.load(), dates)\n            observed = observed.load()\n            lat, lon = observed.lat.values, observed.lon.values\n            require(np.array_equal(lat, np.arange(-60, 15.25, .25)) and np.array_equal(lon, np.arange(-90, -24.75, .25)), \'Unexpected target grid.\')\n            regions = {\'full_domain\': np.ones(len(lat), bool), \'south_of_35S\': lat < -35,\n                       \'35S_to_15S\': (lat >= -35) & (lat < -15), \'north_of_15S\': lat >= -15}\n            original_names = dict(full_domain=\'dominio_inteiro\', south_of_35S=\'lat_menor_que_menos35\',\n                                  **{\'35S_to_15S\': \'lat_menos35_a_menos15\', \'north_of_15S\': \'lat_maior_igual_menos15\'})\n            if pixel_sums is None:\n                pixel_sums = {model: {metric: np.zeros((len(lat), len(lon))) for metric in [\'sse\', \'error\', \'absolute_error\']} for model in MODELS}\n            for i, date in enumerate(dates):\n                truth = observed.values[i].astype(\'float64\')\n                masks = intensity_masks(truth)\n                for model in MODELS:\n                    pred = ds[model].values[i].astype(\'float64\')\n                    error = pred - truth\n                    pixel_sums[model][\'sse\'] += error ** 2\n                    pixel_sums[model][\'error\'] += error\n                    pixel_sums[model][\'absolute_error\'] += np.abs(error)\n                    for region, mask in regions.items():\n                        row = stats(truth[mask], pred[mask])\n                        old = archived[(archived.modelo == model) & (archived.mes_alvo == str(date.date())) & (archived.regiao == original_names[region])]\n                        require(len(old) == 1, \'Missing or duplicated archived monthly score.\')\n                        old = old.iloc[0]\n                        for new_key, old_key in [(\'n\', \'n\'), (\'sse\', \'sse\'), (\'error_sum\', \'soma_erro\'), (\'absolute_error_sum\', \'soma_erro_absoluto\')]:\n                            require(np.isclose(row[new_key], old[old_key], rtol=1e-10, atol=1e-8), f\'Recomputed score differs: {block}/{model}/{date}/{region}/{new_key}\')\n                        monthly_rows.append(dict(block=block, date=str(date.date()), calendar_month=date.month, region=region, model=model, **row))\n                    for label, mask in zip(BIN_NAMES, masks):\n                        intensity_rows.append(dict(block=block, model=model, intensity=label, **stats(truth[mask], pred[mask])))\n        inner = pd.DataFrame(read_json(evidence / block / \'inner/training_history.json\'))\n        outer = pd.DataFrame(read_json(evidence / block / \'unet/training_history.json\'))\n        chosen = read_json(evidence / block / \'inner/training.json\')[\'selected_epochs\']\n        require(chosen == int(inner.loc[inner.inner_rmse.idxmin(), \'epoch\']) == len(outer), \'Epoch selection/history mismatch.\')\n        require(chosen == read_json(evidence / block / \'unet/training.json\')[\'selected_epochs\'], \'Outer epochs differ from inner selection.\')\n        histories.append((block, inner, outer, chosen))\n        training_rows.append(dict(block=block, selected_epochs=chosen, inner_epochs_run=len(inner),\n                                  first_inner_rmse=float(inner.inner_rmse.iloc[0]), best_inner_rmse=float(inner.inner_rmse.min()),\n                                  last_inner_rmse=float(inner.inner_rmse.iloc[-1]),\n                                  inner_training_mse_first=float(inner.training_mse.iloc[0]), inner_training_mse_last=float(inner.training_mse.iloc[-1])))\n        print(block, \'maps, dates, hashes, archived scores and epoch selection verified\', flush=True)\n    monthly = pd.DataFrame(monthly_rows)\n    domain = monthly.query("region == \'full_domain\'")\n    intensity_months = pd.DataFrame(intensity_rows)\n    tables = {\'global\': pooled(domain, [\'model\']), \'calendar_month\': pooled(domain, [\'model\', \'calendar_month\']),\n              \'region\': pooled(monthly, [\'model\', \'region\']), \'region_calendar_month\': pooled(monthly, [\'model\', \'region\', \'calendar_month\']),\n              \'intensity\': pooled(intensity_months, [\'model\', \'intensity\']), \'block_intensity\': pooled(intensity_months, [\'model\', \'block\', \'intensity\']),\n              \'training\': pd.DataFrame(training_rows)}\n    total_n = int(tables[\'global\'].query("model == \'hybrid\'").n.iloc[0])\n    require(total_n == 168 * 301 * 261, \'Incomplete evaluation.\')\n    np.testing.assert_allclose(tables[\'intensity\'].groupby(\'model\')[SUMS].sum().sort_index(), tables[\'global\'].set_index(\'model\')[SUMS].sort_index(), rtol=1e-12, atol=1e-7)\n    archived_global = pd.read_csv(evidence / \'global.csv\').set_index(\'modelo\')\n    for model in MODELS:\n        now = tables[\'global\'].set_index(\'model\').loc[model]\n        require(np.isclose(now.rmse, archived_global.loc[model, \'rmse\'], rtol=0, atol=1e-12), \'Global score did not reproduce.\')\n    for name, keys in [(\'calendar\', [\'calendar_month\']), (\'region\', [\'region\']), (\'region_calendar\', [\'region\', \'calendar_month\']), (\'intensity\', [\'intensity\']), (\'block_intensity\', [\'block\', \'intensity\'])]:\n        base = {\'calendar\': \'calendar_month\', \'region_calendar\': \'region_calendar_month\'}.get(name, name)\n        tables[name + \'_comparison\'] = comparison(tables[base], keys, total_n)\n    variables = {}\n    for model in MODELS:\n        for name, values in [(\'rmse\', np.sqrt(pixel_sums[model][\'sse\'] / 168)), (\'bias\', pixel_sums[model][\'error\'] / 168), (\'mae\', pixel_sums[model][\'absolute_error\'] / 168)]:\n            variables[model + \'_\' + name] = ((\'lat\', \'lon\'), values, dict(units=\'mm/day\'))\n    maps = xr.Dataset(variables, coords=dict(lat=lat, lon=lon), attrs=dict(months=168, independent_holdout=\'false\', analysis=\'development diagnostics; no fitting\'))\n    better = maps.unet_rmse < maps.hybrid_rmse\n    global_values = tables[\'global\'].set_index(\'model\')\n    summary = dict(months=168, gridpoint_months=total_n, all_seven_prediction_hashes_verified=True,\n                   monthly_region_scores_reproduced=len(monthly), intensity_partition_verified=True,\n                   hybrid_rmse=float(global_values.loc[\'hybrid\', \'rmse\']), unet_rmse=float(global_values.loc[\'unet\', \'rmse\']),\n                   global_delta_mse=float((global_values.loc[\'unet\', \'sse\'] - global_values.loc[\'hybrid\', \'sse\']) / total_n),\n                   unet_lower_rmse_grid_fraction=float(better.mean()), unet_higher_rmse_grid_fraction=float((maps.unet_rmse > maps.hybrid_rmse).mean()),\n                   calendar_months_unet_improved=int((tables[\'calendar_comparison\'].delta_rmse < 0).sum()),\n                   independent_holdout=False, model_fitted=False, model_promoted=False, forecast_issued=False)\n    output.mkdir(parents=True)\n    for name, table in tables.items():\n        table.to_csv(output / (name + \'.csv\'), index=False, lineterminator=\'\\n\')\n    maps.to_netcdf(output / \'error_maps.nc\', engine=\'h5netcdf\')\n    save_json(output / \'summary.json\', summary)\n    plot_results(tables, maps, histories, output)\n    save_json(output / \'verification.json\', dict(inputs=inputs, code_sha256=digest(__file__),\n        python=platform.python_version(), versions={name: importlib.metadata.version(name) for name in [\'numpy\', \'pandas\', \'xarray\', \'matplotlib\', \'h5netcdf\']},\n        historical_run_environment=signature[\'inputs\'][\'versions\'], environment_scope=\'diagnostics only; original model was not rerun\',\n        bins_mm_day=[\'0\', \'1\', \'3\', \'5\', \'10\', \'infinity\'],\n        files={p.name: digest(p) for p in sorted(output.iterdir()) if p.is_file()}))\n    print(json.dumps(summary, indent=2))\n    return summary\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--predictions\', type=Path, required=True, help=\'Directory containing H1/.../C predictions.nc files\')\n    parser.add_argument(\'--observations\', type=Path, required=True, help=\'Original official treino_tp.nc\')\n    parser.add_argument(\'--output\', type=Path, required=True, help=\'New report directory\')\n    args = parser.parse_args()\n    execute(args.predictions, args.observations, args.output)\n', 'research/fixed_blend/evaluate.py': '"""Evaluate one fixed blend after the workflow verifies the archived experiment."""\nfrom pathlib import Path\nimport json\n\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nfrom research.diagnostics.analyze_errors import (\n    BLOCKS, BIN_NAMES, MODELS as ORIGINAL_MODELS, SUMS, digest, exact_alignment,\n    intensity_masks, pooled, require, save_json, stats, verify_file,\n)\n\nPROTOCOL = Path(__file__).with_name(\'protocol.json\')\nMODELS = (\'hybrid\', \'unet\', \'blend\', \'climatology\')\n\n\ndef validate_protocol():\n    protocol = json.loads(PROTOCOL.read_text(encoding=\'utf-8\'))\n    require(protocol[\'id\'] == \'fixed_blend_v1\' and protocol[\'weights\'] == {\'hybrid\': .75, \'unet\': .25},\n            \'This adapter only supports the prespecified 75/25 blend.\')\n    require(protocol[\'weight_search\'] is False and protocol[\'conditional_switching\'] is False\n            and protocol[\'new_training\'] is False, \'Fixed-blend protocol changed.\')\n    require(protocol[\'blocks\'] == list(BLOCKS) and protocol[\'evaluation\'] == [\'2007-01\', \'2020-12\'],\n            \'Fixed-blend evaluation period changed.\')\n    return protocol\n\n\ndef fixed_blend(hybrid, unet):\n    hybrid, unet = np.asarray(hybrid, dtype=\'float64\'), np.asarray(unet, dtype=\'float64\')\n    require(hybrid.shape == unet.shape, \'Blend maps have different shapes.\')\n    require(np.isfinite(hybrid).all() and np.isfinite(unet).all(), \'Blend inputs contain nonfinite values.\')\n    require((hybrid >= 0).all() and (unet >= 0).all(), \'Blend inputs must be reconstructed nonnegative rainfall.\')\n    return .75 * hybrid + .25 * unet\n\n\ndef error_products(observed, hybrid, unet):\n    require(np.shape(observed) == np.shape(hybrid) == np.shape(unet), \'Unpaired error products.\')\n    h = np.asarray(hybrid, dtype=\'float64\') - np.asarray(observed, dtype=\'float64\')\n    u = np.asarray(unet, dtype=\'float64\') - np.asarray(observed, dtype=\'float64\')\n    return dict(n=h.size, hybrid_error_sum=float(h.sum()), unet_error_sum=float(u.sum()),\n                hybrid_sse=float(np.square(h).sum()), unet_sse=float(np.square(u).sum()),\n                error_product_sum=float((h * u).sum()), disagreement_sse=float(np.square(u - h).sum()))\n\n\ndef summarize_products(rows):\n    result = {key: sum(row[key] for row in rows) for key in rows[0]}\n    n = result[\'n\']\n    h2, u2, cross = result[\'hybrid_sse\'], result[\'unet_sse\'], result[\'error_product_sum\']\n    hvar = max(0., h2 - result[\'hybrid_error_sum\'] ** 2 / n)\n    uvar = max(0., u2 - result[\'unet_error_sum\'] ** 2 / n)\n    covariance_sum = cross - result[\'hybrid_error_sum\'] * result[\'unet_error_sum\'] / n\n    result[\'centered_error_correlation\'] = float(covariance_sum / np.sqrt(hvar * uvar)) if hvar * uvar > 0 else None\n    result[\'blend_sse_from_products\'] = .75 ** 2 * h2 + .25 ** 2 * u2 + 2 * .75 * .25 * cross\n    # This identity tests the fixed combination, without estimating an optimal weight.\n    result[\'blend_sse_from_disagreement\'] = .75 * h2 + .25 * u2 - .75 * .25 * result[\'disagreement_sse\']\n    return result\n\n\ndef paired(table, keys):\n    baseline = table[table.model == \'hybrid\'].set_index(keys)\n    candidate = table[table.model == \'blend\'].set_index(keys).reindex(baseline.index)\n    require(np.array_equal(candidate.n, baseline.n), \'Unpaired blend comparison.\')\n    result = baseline[[\'n\', \'rmse\', \'mae\', \'bias\']].rename(columns={k: \'hybrid_\' + k for k in [\'rmse\', \'mae\', \'bias\']})\n    for metric in [\'rmse\', \'mae\', \'bias\']:\n        result[\'blend_\' + metric] = candidate[metric]\n        result[\'delta_\' + metric] = candidate[metric] - baseline[metric]\n    result[\'delta_absolute_bias\'] = candidate.bias.abs() - baseline.bias.abs()\n    result[\'delta_sse\'] = candidate.sse - baseline.sse\n    return result.reset_index()\n\n\ndef execute(predictions, observations, output, diagnostics, evidence):\n    """Run only on maps already checked by the original diagnostic in this run."""\n    predictions, observations, output, diagnostics, evidence = map(Path, (predictions, observations, output, diagnostics, evidence))\n    require(not output.exists(), \'Choose a fresh fixed-blend output directory.\')\n    protocol = validate_protocol()\n    verified = json.loads((diagnostics / \'verification.json\').read_text(encoding=\'utf-8\'))\n    verify_file(observations, verified[\'inputs\'][\'observations\'])\n    output.mkdir(parents=True)\n    save_json(output / \'protocol.json\', protocol)\n    monthly, intensities, products = [], [], []\n    pixel_sums = None\n    for index, block in enumerate(BLOCKS):\n        path = predictions / block / \'predictions.nc\'\n        verify_file(path, verified[\'inputs\'][block + \'/predictions.nc\'])\n        dates = pd.date_range(f\'{2007 + 2 * index}-01-01\', periods=24, freq=\'MS\')\n        with xr.open_dataset(observations) as source, xr.open_dataset(path) as dataset:\n            require(source.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected rainfall units.\')\n            observed, dataset = exact_alignment(source.tp, dataset.load(), dates)\n            observed = observed.load()\n            lat, lon = observed.lat.values, observed.lon.values\n            blend = fixed_blend(dataset.hybrid.values, dataset.unet.values)\n            candidate = xr.Dataset({\'blend\': ((\'time\', \'lat\', \'lon\'), blend, {\'units\': \'mm/day\'})},\n                                   coords=dataset.coords,\n                                   attrs=dict(protocol=\'fixed_blend_v1\', hybrid_weight=.75, unet_weight=.25,\n                                              source_prediction_sha256=digest(path), independent_holdout=\'false\',\n                                              scope=\'historical development; not a new operational forecast\'))\n            target = output / \'predictions\' / block / \'predictions.nc\'\n            target.parent.mkdir(parents=True)\n            candidate.to_netcdf(target, engine=\'h5netcdf\')\n            with xr.open_dataset(target) as written:\n                np.testing.assert_array_equal(written.blend.values, blend)\n                require(written.blend.dtype == np.dtype(\'float64\'), \'Blend precision changed on disk.\')\n            regions = {\'full_domain\': np.ones(len(lat), bool), \'south_of_35S\': lat < -35,\n                       \'35S_to_15S\': (lat >= -35) & (lat < -15), \'north_of_15S\': lat >= -15}\n            if pixel_sums is None:\n                pixel_sums = {model: {key: np.zeros((len(lat), len(lon))) for key in [\'sse\', \'error\', \'absolute_error\']} for model in MODELS}\n            for i, date in enumerate(dates):\n                truth = observed.values[i].astype(\'float64\')\n                masks = intensity_masks(truth)\n                product = error_products(truth, dataset.hybrid.values[i], dataset.unet.values[i])\n                product.update(block=block, date=str(date.date()))\n                products.append(product)\n                for model in MODELS:\n                    pred = blend[i] if model == \'blend\' else dataset[model].values[i].astype(\'float64\')\n                    error = pred - truth\n                    for metric, value in [(\'sse\', error ** 2), (\'error\', error), (\'absolute_error\', np.abs(error))]:\n                        pixel_sums[model][metric] += value\n                    for region, mask in regions.items():\n                        monthly.append(dict(block=block, date=str(date.date()), year=date.year,\n                                            calendar_month=date.month, region=region, model=model,\n                                            **stats(truth[mask], pred[mask])))\n                    for label, mask in zip(BIN_NAMES, masks):\n                        intensities.append(dict(block=block, model=model, intensity=label, **stats(truth[mask], pred[mask])))\n        print(block, \'fixed 75/25 maps saved and read back; paired errors calculated\', flush=True)\n\n    monthly = pd.DataFrame(monthly)\n    domain = monthly[monthly.region == \'full_domain\']\n    intensity_rows = pd.DataFrame(intensities)\n    tables = {\'global\': pooled(domain, [\'model\']), \'blocks\': pooled(domain, [\'model\', \'block\']),\n              \'years\': pooled(domain, [\'model\', \'year\']), \'calendar\': pooled(domain, [\'model\', \'calendar_month\']),\n              \'region\': pooled(monthly, [\'model\', \'region\']), \'intensity\': pooled(intensity_rows, [\'model\', \'intensity\']),\n              \'block_intensity\': pooled(intensity_rows, [\'model\', \'block\', \'intensity\'])}\n    values = tables[\'global\'].set_index(\'model\')\n    require((values.n == 168 * 301 * 261).all(), \'Incomplete comparison.\')\n    original = pd.read_csv(diagnostics / \'global.csv\').set_index(\'model\')\n    np.testing.assert_allclose(values.loc[list(ORIGINAL_MODELS), SUMS], original.loc[list(ORIGINAL_MODELS), SUMS], rtol=1e-12, atol=1e-7)\n    np.testing.assert_allclose(tables[\'intensity\'].groupby(\'model\')[SUMS].sum().sort_index(), values[SUMS].sort_index(), rtol=1e-12, atol=1e-7)\n    for name, keys in [(\'blocks\', [\'block\']), (\'years\', [\'year\']), (\'calendar\', [\'calendar_month\']),\n                       (\'region\', [\'region\']), (\'intensity\', [\'intensity\']), (\'block_intensity\', [\'block\', \'intensity\'])]:\n        tables[name + \'_comparison\'] = paired(tables[name], keys)\n    numerical_products = [{key: value for key, value in row.items() if key not in {\'block\', \'date\'}} for row in products]\n    complementarity = summarize_products(numerical_products)\n    np.testing.assert_allclose([complementarity[\'blend_sse_from_products\'], complementarity[\'blend_sse_from_disagreement\']],\n                               values.loc[\'blend\', \'sse\'], rtol=1e-12, atol=1e-7)\n    maps = xr.Dataset({f\'{model}_{metric}\': ((\'lat\', \'lon\'), array, {\'units\': \'mm/day\'})\n                       for model in MODELS for metric, array in [\n                           (\'rmse\', np.sqrt(pixel_sums[model][\'sse\'] / 168)),\n                           (\'mae\', pixel_sums[model][\'absolute_error\'] / 168),\n                           (\'bias\', pixel_sums[model][\'error\'] / 168)]}, coords=dict(lat=lat, lon=lon),\n                      attrs=dict(months=168, protocol=\'fixed_blend_v1\', independent_holdout=\'false\'))\n    summary = dict(protocol=\'fixed_blend_v1\', months=168, gridpoint_months=int(values.loc[\'blend\', \'n\']),\n                   hybrid_rmse=float(values.loc[\'hybrid\', \'rmse\']), unet_rmse=float(values.loc[\'unet\', \'rmse\']),\n                   blend_rmse=float(values.loc[\'blend\', \'rmse\']),\n                   delta_rmse_vs_hybrid=float(values.loc[\'blend\', \'rmse\'] - values.loc[\'hybrid\', \'rmse\']),\n                   relative_rmse_reduction_percent=float(100 * (1 - values.loc[\'blend\', \'rmse\'] / values.loc[\'hybrid\', \'rmse\'])),\n                   delta_mae_vs_hybrid=float(values.loc[\'blend\', \'mae\'] - values.loc[\'hybrid\', \'mae\']),\n                   delta_absolute_bias_vs_hybrid=float(abs(values.loc[\'blend\', \'bias\']) - abs(values.loc[\'hybrid\', \'bias\'])),\n                   blend_lower_rmse_grid_fraction=float((maps.blend_rmse < maps.hybrid_rmse).mean()),\n                   input_hashes_verified=True, original_scores_reproduced=True, blend_maps_read_back_exactly=True,\n                   mse_identity_verified=True, intensity_partition_verified=True, independent_holdout=False,\n                   model_fitted=False, weight_search=False, model_promoted=False, forecast_issued=False)\n    for name in [\'blocks\', \'years\', \'calendar\']:\n        delta = tables[name + \'_comparison\'].delta_rmse\n        summary[name] = dict(improved=int((delta < 0).sum()), worsened=int((delta > 0).sum()), tied=int((delta == 0).sum()))\n    for name, table in tables.items():\n        table.to_csv(output / (name + \'.csv\'), index=False, lineterminator=\'\\n\')\n    monthly.to_csv(output / \'monthly_metrics.csv\', index=False, lineterminator=\'\\n\')\n    pd.DataFrame(products).to_csv(output / \'monthly_error_products.csv\', index=False, lineterminator=\'\\n\')\n    maps.to_netcdf(output / \'error_maps.nc\', engine=\'h5netcdf\')\n    save_json(output / \'complementarity.json\', complementarity)\n    save_json(output / \'summary.json\', summary)\n    from .report import plot_results\n    plot_results(tables, maps, output)\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n', 'research/sst_extension/evidence/completed_20260930/evaluation/global.csv': 'modelo,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area,rmse,mae,vies,rmse_area\nclimatology,1885464,6636712.305557575,93262.72383335876,2132920.31411643,6277256.8181300685,1619418.8409421304,1.8761492047197617,1.1312442529353146,0.04946407029429295,1.9688170181230564\nhybrid,1885464,6087091.083042482,241685.94897880653,2055483.674319979,5745872.031969661,1619418.8409421304,1.7967836118898637,1.0901739170410991,0.1281838046119186,1.8836420669177203\nhybrid_sst,1885464,6059004.250388479,257272.26331628993,2050436.940174953,5718955.130673785,1619418.8409421304,1.7926334924671175,1.087497263365916,0.13645037153522419,1.8792248674258742\n', 'research/sst_extension/evidence/completed_20260930/frozen.json': '{\n  "evaluation_labels_read": false,\n  "files": {\n    "auditoria_CFSv2_desenvolvimento.json": "153a8e20b230164a510de006c6f905d8b17e71e57d1737950a472dd17858f9be",\n    "auditoria_CFSv2_somente_ajuste_final.json": "45da279275deeee3e4152f34ebdbeb9113da24356ac46110f0cc5250b202084c",\n    "code/competition/metadata/NOAA/indices_noaa.csv": "b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4",\n    "code/research/common/inputs.py": "d22140d4ad3db4ac3fecdca26c7ebdd1d780486b51622a1f333fce2a637e6851",\n    "code/research/lagged_sources/library.py": "4b92dd77d868d7806fd0ae37d2837abfcde36e173df75b1bd6d88760593ee9c3",\n    "code/research/lagged_sources/ocean_indices.csv": "69226b2e65f783b4d4971a254d1e325d0908447c76ec1df3bf532ecef1f0cf5b",\n    "code/research/lagged_sources/official_hashes.json": "bd5648197c4cf25f85d89a32c701b422f1e0b788421766cc7b8aca99ea42223d",\n    "code/research/sst_extension/__init__.py": "973626224642d88e2a717b1a8571dace2592c939d1fc9160350568b2facb1533",\n    "code/research/sst_extension/data.py": "2d4a5be5f4e35f1fa9782485f50d6be99e16653348f52b08deec584e6c38d7e9",\n    "code/research/sst_extension/experiment.py": "a9d4597e38d090c33dcc61c5e46f18d5a725d3aa1b013b0a17dd7cf9758243d7",\n    "code/research/sst_extension/protocol.json": "16d6d5a26ea621bc0a702b7b31c17b7b8f1ee0a9d9d97397196d3102f08b9be3",\n    "code/research/sst_extension/test_sst.py": "b604b786779ccb2dc69899543c31f2a3b6212a12ee7fa6ff3e01bac13cd0c0e8",\n    "code/research/temporal_extension/evidence/kaggle_354026629/evaluation/global.csv": "044c310fe49c905b01b4963704cc550c31da6a9ac0bdaaf531361e569a378f36",\n    "indices.json": "607a983c219d9ba8b1d8771dea353404d31525d7cd7a6ef1d4abc2b2f3ebca3e",\n    "input_audit.json": "943a7dd17178ae7a938cd64cb64f007c8a710829b7700f74f22961ff449082e5",\n    "models/climate_atmosphere_indices.npz": "06e2a3cfcceeb2e606e35c76a5903aeac87b9930577d0e86d510781096596f17",\n    "models/climate_cfsv2.nc": "8ac437c945540d2695f4cbe05a5ba9d486da0677a36c1a8bd6562b30449224f5",\n    "models/climate_rainfall.nc": "840cf1fc0491a46fe2d663079af281e50c18a5f6e06c1ae54ef5298e1a2e77c3",\n    "models/climate_seas5.nc": "23217e622dff77627782121bdfa87ca39f0e32e6159289530cdc055f50ac0847",\n    "models/fit.json": "9cb969acf8a88209e102e5352ace59afd35fd7ef57930339d53adb2d52d95efc",\n    "models/forecast_pcs.csv": "9f5c158b07334b93cded9e71d774e10cf7379f22bcb714e05e50f7f0fbaa1f58",\n    "models/hybrid_arvores_multissistema.txt": "36d8562ceee3dcbe22960b4bd6f1ef889727bbe2c4b8e8116d8db7c400563b66",\n    "models/hybrid_arvores_seas5.txt": "5f7891d2107394c763cde96574c2c37b4f116d88f1b06feeeeaa01ed23f645f5",\n    "models/hybrid_sst_arvores_multissistema.txt": "cdc2563032033078ea39e7a0249d42d1027557b51240c06ad270cb3e454c5d25",\n    "models/hybrid_sst_arvores_seas5.txt": "833c4d3b454a3232ef2ac81db534146d038e913a3e15d56a2e6e91e8ea048d1e",\n    "models/pca.npz": "38bc8d26cd619762ebff1540209bae86d893094ae11cedab9dca496ed66cdd90",\n    "models/predictions.nc": "8888ce1301a389c98ac832dfa9d353d8611c1f1c26910d011a18ad19faf7c7cf",\n    "models/ridge.npz": "d6f66fb933c837aa60611aa91f442bae843fd610012fbd113cb5c5cf74b7fcfe",\n    "models/sample.npz": "d65c7b0d36b8aa82c72875c8a7f1ad599ce01c5137b77123c807cf5b403f7ce9",\n    "plan.json": "230406bd2bd32b5cbd7018a496f4b2540391975b975cd432c615178ffe6b5265",\n    "signature.json": "2a28304099751e37ac9c8137528e457beb0eee45fdcdbf32d19e4aff9b4f2b00",\n    "source_calendar.csv": "2c1b612ef95d7b07ab534bec8e5e08b0491c443ab7895c6d30ed5992e67231c9"\n  },\n  "forecast_issued": false,\n  "frozen_at": "2026-09-30T16:27:13.458723+00:00"\n}\n', 'research/sst_unet_blend/__init__.py': '"""One fixed combination of archived SST-hybrid and U-Net predictions."""\n', 'research/sst_unet_blend/experiment.py': '"""Evaluate a single fixed combination from saved maps; no model fitting is imported."""\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom time import perf_counter\nimport base64\nimport json\nimport platform\nimport shutil\nimport zipfile\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom research.common.inputs import ROOT, require, sha256, write_json\nfrom research.fixed_blend.evaluate import fixed_blend, error_products, summarize_products\n\nPROTOCOL = Path(__file__).with_name(\'protocol.json\')\nSST_EVIDENCE = \'research/sst_extension/evidence/completed_20260930\'\nUNET_EVIDENCE = \'research/temporal_extension/evidence/kaggle_354026629\'\nMODELS = (\'hybrid\', \'hybrid_sst\', \'unet\', \'blend_reference\', \'blend_sst\', \'climatology\')\nSUMS = [\'n\', \'sse\', \'error_sum\', \'absolute_error_sum\', \'sse_area\', \'weight_area\']\n\n\ndef protocol():\n    plan = json.loads(PROTOCOL.read_text())\n    require(plan[\'id\'] == \'sst_unet_fixed_blend_2021_2022_v1\'\n            and plan[\'weights\'] == [.75, .25] and plan[\'training_end\'] == \'2020-09-01\', \'Fixed experiment changed.\')\n    require(plan[\'evaluation_start\'] == \'2021-01-01\' and plan[\'evaluation_end\'] == \'2022-12-01\', \'Evaluation dates changed.\')\n    require(not any(plan[k] for k in [\'new_training\', \'weight_search\', \'conditional_switching\', \'automatic_promotion\']),\n            \'This experiment evaluates saved maps at fixed weights only.\')\n    # The source receipts establish the intended runs; no similarly named experiment is substituted.\n    sst = json.loads((ROOT / SST_EVIDENCE / \'frozen.json\').read_text())\n    unet = json.loads((ROOT / UNET_EVIDENCE / \'frozen.json\').read_text())\n    require(plan[\'inputs\'][\'sst_predictions\'][\'sha256\'] == sst[\'files\'][\'models/predictions.nc\']\n            and plan[\'inputs\'][\'unet_predictions\'][\'sha256\'] == unet[\'files\'][\'predictions.nc\']\n            and plan[\'inputs\'][\'observations\'][\'sha256\'] == unet[\'observation_file_sha256\'], \'Pinned source hashes differ.\')\n    return plan\n\n\ndef locate_files(plan, supplied=None, roots=None):\n    """Find exact bytes, allowing renamed files when their explicit paths are supplied."""\n    supplied = supplied or {}\n    roots = list(map(Path, roots or [\'/kaggle/input\', \'/kaggle/working\']))\n    found, hashes = {}, {}\n    for name, spec in plan[\'inputs\'].items():\n        if supplied.get(name):\n            candidate = Path(supplied[name]).resolve()\n            require(candidate.is_file(), f\'Missing {name}: {candidate}\')\n            candidates = [candidate]\n        else:\n            candidates = sorted({p.resolve() for root in roots if root.is_dir()\n                                 for p in root.rglob(spec[\'filename\']) if p.is_file()})\n        matches = []\n        for p in candidates:\n            if p not in hashes:\n                hashes[p] = sha256(p)\n            if hashes[p] == spec[\'sha256\']:\n                matches.append(p)\n        require(matches, f\'Missing exact {name} ({spec["filename"]}). Attach the complete saved output, \'\n                f\'not the reports ZIP. Expected SHA-256 {spec["sha256"]}. Use a file path override if renamed.\')\n        found[name] = matches[0]\n        print(\'Verified:\', name, found[name], flush=True)\n    return found\n\n\ndef validate_maps(sst, unet, dates, require_full_grid=True):\n    require(set(sst.data_vars) == {\'hybrid\', \'hybrid_sst\', \'climatology\'}\n            and set(unet.data_vars) == {\'hybrid\', \'unet\', \'blend\', \'climatology\'}, \'Wrong source-model variables.\')\n    sst, unet = xr.align(sst, unet, join=\'exact\')\n    for ds in [sst, unet]:\n        require(pd.DatetimeIndex(ds.time.values).equals(dates), \'Wrong, missing or reordered dates.\')\n        if require_full_grid:\n            require(np.array_equal(ds.lat, np.arange(-60, 15.25, .25))\n                    and np.array_equal(ds.lon, np.arange(-90, -24.75, .25)), \'Wrong spatial grid.\')\n        for name in ds:\n            field = ds[name]\n            require(field.dims == (\'time\', \'lat\', \'lon\') and field.attrs.get(\'units\') == \'mm/day\', \'Wrong dimensions or units.\')\n            require(np.isfinite(field.values).all() and (field.values >= 0).all(), \'Nonfinite or negative input rainfall.\')\n    for name in [\'hybrid\', \'climatology\']:\n        require(np.array_equal(sst[name].values, unet[name].values), f\'{name} maps differ between source experiments.\')\n    reconstructed = fixed_blend(unet.hybrid.values, unet.unet.values)\n    require(np.array_equal(reconstructed, unet.blend.values), \'Archived reference blend does not reproduce exactly.\')\n    return sst, unet\n\n\ndef combine(sst, unet, dates, require_full_grid=True):\n    sst, unet = validate_maps(sst, unet, dates, require_full_grid)\n    candidate = fixed_blend(sst.hybrid_sst.values, unet.unet.values)\n    return xr.Dataset({\n        \'hybrid\': sst.hybrid.astype(\'float64\'), \'hybrid_sst\': sst.hybrid_sst.astype(\'float64\'),\n        \'unet\': unet.unet.astype(\'float64\'), \'blend_reference\': unet.blend.astype(\'float64\'),\n        \'blend_sst\': xr.DataArray(candidate, dims=(\'time\', \'lat\', \'lon\'), coords=sst.hybrid.coords,\n                                attrs=dict(units=\'mm/day\')),\n        \'climatology\': sst.climatology.astype(\'float64\')},\n        attrs=dict(scope=\'Retrospective 2021-2022 development; no operational issuance\', weights=\'0.75,0.25\'))\n\n\ndef pooled(rows, keys):\n    out = rows.groupby(keys, as_index=False)[SUMS].sum()\n    out[\'rmse\'] = np.sqrt(out.sse / out.n)\n    out[\'mae\'] = out.absolute_error_sum / out.n\n    out[\'bias\'] = out.error_sum / out.n\n    out[\'rmse_area\'] = np.sqrt(out.sse_area / out.weight_area)\n    return out\n\n\ndef evaluate_maps(predictions, truth):\n    truth, predictions = xr.align(truth, predictions, join=\'exact\')\n    require(truth.dims == (\'time\', \'lat\', \'lon\') and truth.attrs.get(\'units\') == \'mm/day\', \'Wrong observation dimensions/units.\')\n    require(set(predictions.data_vars) == set(MODELS), \'Incomplete candidate/reference set.\')\n    require(np.isfinite(truth.values).all() and (truth.values >= 0).all(), \'Invalid observed rainfall.\')\n    lat = truth.lat.values\n    masks = dict(full_domain=np.ones(len(lat), bool), south_of_35S=lat < -35,\n                 between_35S_and_15S=(lat >= -35) & (lat < -15), north_of_15S=lat >= -15)\n    rows, products = [], {\'reference\': [], \'sst\': []}\n    for i, t in enumerate(pd.DatetimeIndex(truth.time.values)):\n        observed = truth.values[i].astype(\'float64\')\n        for label, hybrid in [(\'reference\', \'hybrid\'), (\'sst\', \'hybrid_sst\')]:\n            products[label].append(error_products(observed, predictions[hybrid].values[i], predictions.unet.values[i]))\n        for name in MODELS:\n            pred = predictions[name].values[i].astype(\'float64\')\n            require(np.isfinite(pred).all() and (pred >= 0).all(), \'Invalid prediction values.\')\n            error = pred - observed\n            for region, mask in masks.items():\n                if not mask.any():\n                    continue\n                e = error[mask]\n                area = np.broadcast_to(np.cos(np.deg2rad(lat[mask]))[:, None], e.shape)\n                rows.append(dict(model=name, date=str(t.date()), year=t.year, month=t.month, region=region,\n                            n=e.size, sse=float(np.square(e).sum()), error_sum=float(e.sum()),\n                            absolute_error_sum=float(np.abs(e).sum()), sse_area=float((area * e**2).sum()), weight_area=float(area.sum())))\n    detail = pd.DataFrame(rows)\n    whole = detail[detail.region == \'full_domain\']\n    tables = dict(monthly_metrics=detail, global_metrics=pooled(whole, [\'model\']),\n                  years=pooled(whole, [\'model\', \'year\']), months=pooled(whole, [\'model\', \'date\']),\n                  regions=pooled(detail, [\'model\', \'region\']), calendar=pooled(whole, [\'model\', \'month\']))\n    products = {k: summarize_products(v) for k, v in products.items()}\n    g = tables[\'global_metrics\'].set_index(\'model\')\n    for key, model in [(\'reference\', \'blend_reference\'), (\'sst\', \'blend_sst\')]:\n        np.testing.assert_allclose([products[key][\'blend_sse_from_products\'], products[key][\'blend_sse_from_disagreement\']],\n                                   g.loc[model, \'sse\'], rtol=1e-12, atol=1e-7)\n    return tables, products\n\n\ndef compare_references(table):\n    current = table.set_index(\'model\')\n    deviations = {}\n    for folder, mapping in [(SST_EVIDENCE, {\'hybrid\': \'hybrid\', \'hybrid_sst\': \'hybrid_sst\', \'climatology\': \'climatology\'}),\n                            (UNET_EVIDENCE, {\'hybrid\': \'hybrid\', \'unet\': \'unet\', \'blend\': \'blend_reference\', \'climatology\': \'climatology\'})]:\n        archived = pd.read_csv(ROOT / folder / \'evaluation/global.csv\').set_index(\'modelo\')\n        for old, new in mapping.items():\n            delta = {k: float(current.loc[new, k] - archived.loc[old, \'vies\' if k == \'bias\' else k])\n                     for k in [\'rmse\', \'mae\', \'bias\', \'rmse_area\']}\n            require(all(abs(v) < 1e-10 for v in delta.values()), \'Archived scores not reproduced: \' + new)\n            require(int(current.loc[new, \'n\']) == int(archived.loc[old, \'n\']), \'Different reference counts.\')\n            deviations[f\'{folder}/{old}\'] = delta\n    return deviations\n\n\ndef source_files():\n    files = [\'research/common/inputs.py\', \'research/diagnostics/analyze_errors.py\', \'research/fixed_blend/evaluate.py\',\n             \'research/sst_unet_blend/protocol.json\']\n    for folder in [SST_EVIDENCE, UNET_EVIDENCE]:\n        files.extend([folder + \'/frozen.json\', folder + \'/evaluation/global.csv\'])\n    files += [p.relative_to(ROOT).as_posix() for p in Path(__file__).parent.glob(\'*.py\')]\n    return sorted(set(files))\n\n\ndef utcnow():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef execute(output, observations=None, sst_predictions=None, unet_predictions=None):\n    started = perf_counter()\n    plan = protocol()\n    inputs = locate_files(plan, dict(observations=observations, sst_predictions=sst_predictions, unet_predictions=unet_predictions))\n    output = Path(output).resolve()\n    signature = dict(plan=plan, input_hashes={k: sha256(p) for k, p in inputs.items()},\n                     code={n: sha256(ROOT / n) for n in source_files()},\n                     versions=dict(numpy=np.__version__, pandas=pd.__version__, xarray=xr.__version__, python=platform.python_version()))\n    if (output / \'complete.json\').exists():\n        record = json.loads((output / \'complete.json\').read_text())\n        require(json.loads((output / \'signature.json\').read_text()) == signature, \'Different inputs, code or environment; select a fresh output.\')\n        verify_inventory(output, record[\'files\'])\n        print(\'Complete saved comparison verified; no recalculation.\', flush=True)\n        return record[\'summary\']\n    require(not output.exists(), \'An incomplete output exists. Preserve it and choose a new output directory.\')\n    require(not output.is_relative_to(ROOT / \'research\') and not output.is_relative_to(ROOT / \'competition\'), \'Output must be separate from code.\')\n    require(all(not p.is_relative_to(output) for p in inputs.values()), \'An input is inside the output directory.\')\n    output.mkdir(parents=True)\n    write_json(output / \'signature.json\', signature)\n    for name in signature[\'code\']:\n        dest = output / \'code\' / name\n        dest.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(ROOT / name, dest)\n        require(sha256(dest) == signature[\'code\'][name], \'Code changed during snapshot.\')\n    dates = pd.date_range(plan[\'evaluation_start\'], plan[\'evaluation_end\'], freq=\'MS\')\n    with xr.open_dataset(inputs[\'sst_predictions\']) as a, xr.open_dataset(inputs[\'unet_predictions\']) as b:\n        predictions = combine(a.load(), b.load(), dates)\n    predictions.to_netcdf(output / \'predictions.nc\', engine=\'h5netcdf\',\n                           encoding={n: dict(zlib=True, complevel=4) for n in predictions})\n    with xr.open_dataset(output / \'predictions.nc\') as saved:\n        for n in predictions:\n            np.testing.assert_array_equal(saved[n].values, predictions[n].values)\n            require(saved[n].dtype == np.dtype(\'float64\'), \'Output precision changed.\')\n    write_json(output / \'candidate_frozen.json\', dict(frozen_at=utcnow(), sha256=sha256(output / \'predictions.nc\'),\n               evaluation_labels_opened=False, independent_holdout=False, forecast_issued=False))\n    print(\'Source maps verified; candidate written and read back. Opening observations for scoring.\', flush=True)\n    with xr.open_dataset(inputs[\'observations\']) as ds:\n        truth = ds.tp.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').load()\n    tables, products = evaluate_maps(predictions, truth)\n    require((tables[\'global_metrics\'].n == 1885464).all(), \'Incomplete full-grid scoring.\')\n    checks = compare_references(tables[\'global_metrics\'])\n    for name, table in tables.items():\n        table.to_csv(output / (name + \'.csv\'), index=False)\n    write_json(output / \'reference_checks.json\', checks)\n    write_json(output / \'error_products.json\', products)\n    g = tables[\'global_metrics\'].set_index(\'model\')\n    comparisons = {}\n    for reference in [\'blend_reference\', \'hybrid_sst\', \'hybrid\']:\n        entry = {metric: float(g.loc[\'blend_sst\', metric] - g.loc[reference, metric]) for metric in [\'rmse\', \'mae\', \'bias\', \'rmse_area\']}\n        entry[\'absolute_bias\'] = float(abs(g.loc[\'blend_sst\', \'bias\']) - abs(g.loc[reference, \'bias\']))\n        entry[\'rmse_reduction_percent\'] = float(100 * (1 - g.loc[\'blend_sst\', \'rmse\'] / g.loc[reference, \'rmse\']))\n        for name, key in [(\'years\', \'year\'), (\'months\', \'date\')]:\n            paired = tables[name].pivot(index=key, columns=\'model\', values=\'rmse\')\n            delta = paired.blend_sst - paired[reference]\n            entry[name] = dict(better=int((delta < 0).sum()), worse=int((delta > 0).sum()), tied=int((delta == 0).sum()))\n        comparisons[reference] = entry\n    summary = dict(status=\'complete\', complete_comparison=True, complete_seven_blocks=False,\n                   primary_reference=\'blend_reference\', comparisons=comparisons,\n                   archived_scores_reproduced=True, hybrid_maps_identical=True, fixed_blend_identity_verified=True,\n                   model_fits=0, weight_search=False, independent_holdout=False,\n                   forecast_issued=False, model_promoted=False, seconds=perf_counter() - started)\n    write_json(output / \'summary.json\', summary)\n    render_report(output, tables, summary)\n    require(all(sha256(p) == signature[\'input_hashes\'][k] for k, p in inputs.items()), \'An input changed during evaluation.\')\n    frozen = json.loads((output / \'candidate_frozen.json\').read_text())\n    require(sha256(output / \'predictions.nc\') == frozen[\'sha256\'], \'Candidate changed during evaluation.\')\n    files = {p.relative_to(output).as_posix(): sha256(p) for p in sorted(output.rglob(\'*\'))\n             if p.is_file() and \'__pycache__\' not in p.parts}\n    write_json(output / \'complete.json\', dict(files=files, summary=summary))\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n\n\ndef verify_inventory(output, files):\n    output = Path(output).resolve()\n    for name, digest in files.items():\n        p = (output / name).resolve()\n        require(p.is_relative_to(output) and p.is_file() and sha256(p) == digest, \'Changed or missing artifact: \' + name)\n\n\ndef render_report(output, tables, summary):\n    import matplotlib.pyplot as plt\n    monthly = tables[\'months\'].pivot(index=\'date\', columns=\'model\', values=\'rmse\')\n    fig, axes = plt.subplots(2, 1, figsize=(11, 6), layout=\'constrained\')\n    for ax, reference, title in zip(axes, [\'blend_reference\', \'hybrid_sst\'],\n                                    [\'Contribution of SST to the existing hybrid/U-Net blend\', \'Contribution of U-Net to the SST hybrid\']):\n        delta = monthly.blend_sst - monthly[reference]\n        ax.bar(range(24), delta, color=[\'#168293\' if v < 0 else \'#b56343\' for v in delta])\n        ax.axhline(0, color=\'#253746\', lw=.8)\n        ax.set(title=title, ylabel=\'RMSE change (mm/day)\', xticks=range(0, 24, 3),\n               xticklabels=[t[:7] for t in monthly.index[::3]])\n        ax.spines[[\'top\', \'right\']].set_visible(False)\n    fig.savefig(output / \'monthly_comparison.png\', dpi=150)\n    plt.close(fig)\n    encoded = base64.b64encode((output / \'monthly_comparison.png\').read_bytes()).decode()\n    html = \'<!doctype html><html lang="en"><meta charset="utf-8"><title>SST and U-Net fixed blend</title><style>body{font:16px/1.5 system-ui;max-width:1100px;margin:35px auto;padding:0 20px;color:#213744}table{border-collapse:collapse}td,th{padding:7px;border-bottom:1px solid #ddd}img{max-width:100%}</style><body>\'\n    html += \'<h1>Does SST improve the fixed hybrid/U-Net blend?</h1><p>Candidate: 75% SST hybrid + 25% unchanged U-Net. Primary reference: 75% original hybrid + 25% the same U-Net. No models were fitted and no weights were searched.</p>\'\n    html += \'<p>2021–2022 is a previously consulted development period, not an independent holdout. Historical publication vintages remain unverified; no operational model is promoted.</p>\'\n    html += tables[\'global_metrics\'][[\'model\', \'rmse\', \'mae\', \'bias\', \'rmse_area\']].to_html(index=False, float_format=lambda v: f\'{v:.6f}\')\n    html += f\'<img alt="Monthly contribution of SST and U-Net" src="data:image/png;base64,{encoded}">\'\n    html += \'<h2>Annual errors</h2>\' + tables[\'years\'][[\'model\', \'year\', \'rmse\', \'mae\', \'bias\']].to_html(index=False, float_format=lambda v: f\'{v:.6f}\')\n    html += \'<p>All errors use the full supplied grid. Negative changes favor the candidate. Complementarity was checked using both exact fixed-blend MSE identities.</p></body></html>\'\n    (output / \'report.html\').write_text(html, encoding=\'utf-8\')\n\n\ndef export_reports(output):\n    output = Path(output).resolve()\n    record = json.loads((output / \'complete.json\').read_text())\n    verify_inventory(output, record[\'files\'])\n    target = output.parent / \'sst_unet_blend_reports.zip\'\n    with zipfile.ZipFile(target, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for p in sorted(output.rglob(\'*\')):\n            if p.is_file() and p.suffix in {\'.json\', \'.csv\', \'.py\', \'.png\', \'.html\'} and \'__pycache__\' not in p.parts:\n                archive.write(p, p.relative_to(output).as_posix())\n    return target\n', 'research/sst_unet_blend/protocol.json': '{\n  "id": "sst_unet_fixed_blend_2021_2022_v1",\n  "evaluation_start": "2021-01-01",\n  "evaluation_end": "2022-12-01",\n  "training_end": "2020-09-01",\n  "candidate": "0.75 * hybrid_sst + 0.25 * unet",\n  "reference": "0.75 * hybrid + 0.25 * unet",\n  "weights": [0.75, 0.25],\n  "primary_comparison": ["blend_sst", "blend_reference"],\n  "secondary_comparisons": [["blend_sst", "hybrid_sst"], ["blend_sst", "hybrid"]],\n  "arithmetic": "float64 on final nonnegative rainfall maps, no further clipping or calibration",\n  "new_training": false,\n  "new_download_from_data_providers": false,\n  "weight_search": false,\n  "conditional_switching": false,\n  "independent_holdout": false,\n  "prior_use": "2021-2022 was already evaluated for both source experiments; this is a consulted retrospective development block.",\n  "historical_publication_vintages_verified": false,\n  "automatic_promotion": false,\n  "forecast_issued": false,\n  "inputs": {\n    "observations": {"filename": "treino_tp.nc", "sha256": "012bebbe0e38aefdf4f9e2432c3ca22f71d94ea1a1b5421da9d9e5048632cab0"},\n    "sst_predictions": {"filename": "predictions.nc", "sha256": "8888ce1301a389c98ac832dfa9d353d8611c1f1c26910d011a18ad19faf7c7cf", "source": "SST extension completed 2026-09-30: models/predictions.nc"},\n    "unet_predictions": {"filename": "predictions.nc", "sha256": "5e5341220a58a84305d6676ee0d2ceb5fd5dcf7b9b2da3172ad3e3d75050b23f", "source": "Temporal extension Kaggle version 354026629: predictions.nc"}\n  }\n}\n', 'research/sst_unet_blend/test_blend.py': '"""Synthetic numerical, alignment and saved-only workflow checks."""\nfrom contextlib import redirect_stdout\nfrom pathlib import Path\nimport copy\nimport io\nimport json\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom research.common.inputs import ROOT, sha256\nfrom .experiment import (protocol, locate_files, combine, evaluate_maps, compare_references,\n                         execute, export_reports, SST_EVIDENCE, UNET_EVIDENCE)\n\n\ndef maps(full=False):\n    dates = pd.date_range(\'2021-01-01\', periods=24, freq=\'MS\')\n    coords = dict(time=dates, lat=np.arange(-60, 15.25, .25) if full else [-50., -25., 0.],\n                  lon=np.arange(-90, -24.75, .25) if full else [-70., -60.])\n    shape = tuple(len(coords[k]) for k in [\'time\', \'lat\', \'lon\'])\n    def field(value):\n        return xr.DataArray(np.full(shape, value, dtype=\'float64\'), dims=(\'time\', \'lat\', \'lon\'), coords=coords,\n                            attrs=dict(units=\'mm/day\'))\n    h, hs, u, cl = map(field, [2., 2.1, 2.4, 1.8])\n    original = .75 * h + .25 * u\n    original.attrs[\'units\'] = \'mm/day\'\n    return xr.Dataset(dict(hybrid=h, hybrid_sst=hs, climatology=cl)), \\\n        xr.Dataset(dict(hybrid=h, unet=u, blend=original, climatology=cl)), field(2.2), dates\n\n\nclass BlendTests(unittest.TestCase):\n    def test_protocol_pins_two_distinct_completed_runs_and_no_training(self):\n        plan = protocol()\n        self.assertEqual(plan[\'weights\'], [.75, .25])\n        self.assertFalse(plan[\'new_training\'] or plan[\'weight_search\'] or plan[\'automatic_promotion\'])\n        self.assertNotEqual(plan[\'inputs\'][\'sst_predictions\'][\'sha256\'], plan[\'inputs\'][\'unet_predictions\'][\'sha256\'])\n\n    def test_discovery_rejects_same_named_wrong_bytes_and_accepts_renamed_explicit_file(self):\n        with tempfile.TemporaryDirectory() as folder:\n            root = Path(folder)\n            (root / \'a\').mkdir(); (root / \'b\').mkdir()\n            (root / \'a/predictions.nc\').write_bytes(b\'wrong\')\n            good = root / \'b/predictions.nc\'\n            good.write_bytes(b\'right\')\n            plan = dict(inputs=dict(sst_predictions=dict(filename=\'predictions.nc\', sha256=sha256(good))))\n            with redirect_stdout(io.StringIO()):\n                self.assertEqual(locate_files(plan, roots=[root])[\'sst_predictions\'], good.resolve())\n                renamed = root / \'sst_maps.nc\'\n                good.rename(renamed)\n                self.assertEqual(locate_files(plan, dict(sst_predictions=renamed), roots=[root])[\'sst_predictions\'], renamed.resolve())\n            with self.assertRaisesRegex(ValueError, \'Missing exact\'):\n                locate_files(plan, roots=[root])\n\n    def test_fixed_combination_preserves_reference_and_adds_only_sst_change(self):\n        sst, unet, truth, dates = maps()\n        candidate = combine(sst, unet, dates, require_full_grid=False)\n        np.testing.assert_array_equal(candidate.blend_reference, unet.blend)\n        np.testing.assert_allclose(candidate.blend_sst - candidate.blend_reference, .75 * (sst.hybrid_sst - sst.hybrid), atol=1e-15)\n        np.testing.assert_allclose(candidate.blend_sst, 2.175, rtol=0, atol=1e-15)\n        self.assertEqual(candidate.blend_sst.dtype, np.dtype(\'float64\'))\n        tables, products = evaluate_maps(candidate, truth)\n        g = tables[\'global_metrics\'].set_index(\'model\')\n        self.assertAlmostEqual(g.loc[\'blend_sst\', \'rmse\'], .025, places=12)\n        self.assertAlmostEqual(g.loc[\'blend_reference\', \'rmse\'], .1, places=12)\n        self.assertAlmostEqual(g.loc[\'blend_sst\', \'bias\'], -.025, places=12)\n        self.assertAlmostEqual(products[\'sst\'][\'blend_sse_from_products\'], g.loc[\'blend_sst\', \'sse\'], places=10)\n\n    def test_misalignment_missing_dates_units_and_changed_reference_are_rejected(self):\n        sst, unet, _, dates = maps()\n        for invalid in [unet.isel(time=slice(None, -1)), unet.isel(lat=slice(None, None, -1)),\n                        unet.assign_coords(lon=unet.lon + .01)]:\n            with self.assertRaises(ValueError):\n                combine(sst, invalid, dates, require_full_grid=False)\n        for variable in [\'hybrid\', \'climatology\', \'blend\']:\n            invalid = unet.copy(deep=True)\n            invalid[variable].values[0, 0, 0] += .01\n            with self.assertRaises(ValueError):\n                combine(sst, invalid, dates, require_full_grid=False)\n        for value in [-1., np.nan, np.inf]:\n            invalid = sst.copy(deep=True)\n            invalid.hybrid_sst.values[0, 0, 0] = value\n            with self.assertRaises(ValueError):\n                combine(invalid, unet, dates, require_full_grid=False)\n        sst.hybrid_sst.attrs[\'units\'] = \'m\'\n        with self.assertRaisesRegex(ValueError, \'units\'):\n            combine(sst, unet, dates, require_full_grid=False)\n\n    def test_recorded_source_scores_are_required_before_interpretation(self):\n        old = pd.read_csv(ROOT / UNET_EVIDENCE / \'evaluation/global.csv\')\n        new = pd.read_csv(ROOT / SST_EVIDENCE / \'evaluation/global.csv\').query("modelo == \'hybrid_sst\'")\n        table = pd.concat([old, new]).rename(columns=dict(modelo=\'model\', vies=\'bias\'))\n        table.loc[table.model == \'blend\', \'model\'] = \'blend_reference\'\n        self.assertEqual(len(compare_references(table)), 7)\n        table.loc[table.model == \'unet\', \'rmse\'] += .0001\n        with self.assertRaisesRegex(ValueError, \'not reproduced\'):\n            compare_references(table)\n\n    def test_saved_only_end_to_end_export_reuse_and_tamper_detection(self):\n        sst, unet, truth, _ = maps(full=True)\n        with tempfile.TemporaryDirectory() as folder:\n            root = Path(folder)\n            paths = dict(observations=root / \'observations.nc\', sst_predictions=root / \'sst.nc\', unet_predictions=root / \'unet.nc\')\n            for key, data in [(\'observations\', truth.to_dataset(name=\'tp\')), (\'sst_predictions\', sst), (\'unet_predictions\', unet)]:\n                data.to_netcdf(paths[key], engine=\'h5netcdf\')\n            plan = copy.deepcopy(protocol())\n            for name, p in paths.items():\n                plan[\'inputs\'][name][\'sha256\'] = sha256(p)\n            output = root / \'output\'\n            def check_analytic_reference(table):\n                g = table.set_index(\'model\')\n                for name, rmse in [(\'hybrid\', .2), (\'hybrid_sst\', .1), (\'unet\', .2), (\'blend_reference\', .1), (\'climatology\', .4)]:\n                    self.assertAlmostEqual(g.loc[name, \'rmse\'], rmse, places=12)\n                return dict(synthetic=True)\n            with patch(\'research.sst_unet_blend.experiment.protocol\', return_value=plan), \\\n                 patch(\'research.sst_unet_blend.experiment.compare_references\', side_effect=check_analytic_reference), \\\n                 redirect_stdout(io.StringIO()):\n                result = execute(output, **paths)\n                self.assertTrue(result[\'complete_comparison\'])\n                self.assertEqual(result[\'model_fits\'], 0)\n                self.assertEqual(result[\'comparisons\'][\'blend_reference\'][\'months\'][\'better\'], 24)\n                self.assertTrue(export_reports(output).is_file())\n                with patch(\'research.sst_unet_blend.experiment.evaluate_maps\', side_effect=AssertionError(\'Unexpected repeated scoring\')):\n                    self.assertEqual(result, execute(output, **paths))\n                (output / \'summary.json\').write_text(\'{}\')\n                with self.assertRaisesRegex(ValueError, \'Changed or missing\'):\n                    execute(output, **paths)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'research/temporal_extension/evidence/kaggle_354026629/evaluation/global.csv': 'modelo,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area,rmse,mae,vies,rmse_area\nblend,1885464,6076965.304107503,275566.076565667,2060351.1677062258,5735361.502433309,1619418.8409421304,1.7952885294286802,1.0927555061810916,0.14615292393048449,1.8819184697356335\nclimatology,1885464,6636712.305557575,93262.72383335876,2132920.31411643,6277256.8181300685,1619418.8409421304,1.8761492047197617,1.1312442529353146,0.04946407029429295,1.9688170181230564\nhybrid,1885464,6087091.083042482,241685.94897880653,2055483.674319979,5745872.031969661,1619418.8409421304,1.7967836118898637,1.0901739170410991,0.1281838046119186,1.8836420669177203\nunet,1885464,6154974.405279987,377206.4593262484,2099750.103034948,5807365.477739566,1619418.8409421304,1.8067747114937378,1.1136516544653985,0.20006028188618208,1.8936947947541105\n', 'research/temporal_extension/evidence/kaggle_354026629/frozen.json': '{\n  "evaluation_rainfall_decoded": false,\n  "files": {\n    "auditoria_CFSv2_desenvolvimento.json": "153a8e20b230164a510de006c6f905d8b17e71e57d1737950a472dd17858f9be",\n    "auditoria_CFSv2_somente_ajuste_final.json": "45da279275deeee3e4152f34ebdbeb9113da24356ac46110f0cc5250b202084c",\n    "code/competition/metadata/NOAA/indices_noaa.csv": "b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4",\n    "code/research/common/__init__.py": "4f6bfaae6ba45669c7d41e1b087df0582970932f06cdea87a4ff3b352f4135f0",\n    "code/research/common/baseline.py": "653594bf49fa338304129d7d22e7dc31a4e32647b26ee87af7bea4bad94a4dfa",\n    "code/research/common/inputs.py": "d22140d4ad3db4ac3fecdca26c7ebdd1d780486b51622a1f333fce2a637e6851",\n    "code/research/common/synthetic_fixture.py": "da4de7a662c2c8c25f324c5b543ab04914d490ffde5e316c39187eb8f1640bff",\n    "code/research/lagged_sources/library.py": "4b92dd77d868d7806fd0ae37d2837abfcde36e173df75b1bd6d88760593ee9c3",\n    "code/research/lagged_sources/ocean_indices.csv": "69226b2e65f783b4d4971a254d1e325d0908447c76ec1df3bf532ecef1f0cf5b",\n    "code/research/lagged_sources/official_hashes.json": "bd5648197c4cf25f85d89a32c701b422f1e0b788421766cc7b8aca99ea42223d",\n    "code/research/spatial_unet/__init__.py": "ee933fd666f8d33250d76a02fcd8b898d3a2dc7b341dde8165411387cfa27c19",\n    "code/research/spatial_unet/network.py": "f1d10bbe1634018857b4fda8ae2753c1cf43d553f221e338bd1d6dcd89f40654",\n    "code/research/spatial_unet/protocol.json": "b5819403ce177b1c69cfd698160c0e2339f040d5d9ab0a4ad9608a469872c694",\n    "code/research/spatial_unet/run_experiment.py": "4484f2a6c0b29bee18e003ecfab880f0e16e192f1d1b9c71e90bc486f1cd6412",\n    "code/research/spatial_unet/test_spatial.py": "65215314cb5ebd1229ded058c4aab91ab71b9ceccc6af8a83de25668301a8516",\n    "code/research/spatial_unet/training.py": "698d27f397a312c3a4a5f832a15fa09fe431095431b74ca328a8034c3f232275",\n    "code/research/temporal_extension/__init__.py": "73a12c9b4224de54cff998c920ba7d4d20d896ce3a16b89e85ecffb5a36cd9ba",\n    "code/research/temporal_extension/experiment.py": "5172ae29a3575c9bfeb21e5e1396ec09a622a41e34f6fc8dfe7eaf689b9708e7",\n    "code/research/temporal_extension/protocol.json": "367fa9da29f45286c5ce2a8358bf87f415a69d35918c16fc129fffe61c5a8006",\n    "code/research/temporal_extension/report.py": "b9ab84a8091d17618702a7a56a85ca054481dd0268af3db2f56cadc8fcb3d179",\n    "code/research/temporal_extension/test_extension.py": "c83ab2d1fd39d7588613bac3cd85a90e085ee82447236e72209741eaab45085d",\n    "hybrid/arvores_multissistema.txt": "36d8562ceee3dcbe22960b4bd6f1ef889727bbe2c4b8e8116d8db7c400563b66",\n    "hybrid/arvores_seas5.txt": "5f7891d2107394c763cde96574c2c37b4f116d88f1b06feeeeaa01ed23f645f5",\n    "hybrid/clima_atmosfera_indices.npz": "06e2a3cfcceeb2e606e35c76a5903aeac87b9930577d0e86d510781096596f17",\n    "hybrid/clima_cfsv2.nc": "8ac437c945540d2695f4cbe05a5ba9d486da0677a36c1a8bd6562b30449224f5",\n    "hybrid/clima_chuva.nc": "840cf1fc0491a46fe2d663079af281e50c18a5f6e06c1ae54ef5298e1a2e77c3",\n    "hybrid/clima_seas5.nc": "23217e622dff77627782121bdfa87ca39f0e32e6159289530cdc055f50ac0847",\n    "hybrid/fit.json": "56ae0459fa23cbe26be72f89b3949a79d45df2392d08cbf633752e66022ef236",\n    "hybrid/ridge.npz": "d6f66fb933c837aa60611aa91f442bae843fd610012fbd113cb5c5cf74b7fcfe",\n    "indices.json": "de65ad4fb6822e49ed5d543fe66a9ddc83e809fbe284192ba34abf374adb668f",\n    "inner/context.json": "f6a89922df6903165a0905844685750203e18f70a1eac2f864f66e2918aa7b3d",\n    "inner/network.pt": "e2da6ff0d2e700d73cb48b88a8bc18ae1b46b807e0b2a810800668e26868d2a2",\n    "inner/rainfall_climatology.nc": "5ec17987f787c9f2fe7a353dc3aa8330d102b568118123c1e17b50e29757e75d",\n    "inner/references.npz": "347cd4e6ecc8851b64584f8e3a4d9c861458ed82db1ca010ed5ef8bdf52beefb",\n    "inner/scaling.npz": "03f3c24910320c780401cbe4fe83eab695e77045c47f1585c9cdac3be9eae6ed",\n    "inner/training.json": "9965005d6db715030dd6c41e192c0b10bce44906be70c72018706ce1a22f57c1",\n    "inner/training_history.json": "1687441f871fee70e08aba8eb0a55a8954e450d3df05d99687cfdf3203e79606",\n    "plan.json": "843ac4fc8a77aae5850fa8be715a01d3f99b8b298f7ebf04ea5bbf14cad9bd96",\n    "predictions.nc": "5e5341220a58a84305d6676ee0d2ceb5fd5dcf7b9b2da3172ad3e3d75050b23f",\n    "signature.json": "043f83a4f3c23b28b03e540111ec9e9b91f2e3e1b75589f0835e89dff8863f59",\n    "source_calendar.csv": "2c1b612ef95d7b07ab534bec8e5e08b0491c443ab7895c6d30ed5992e67231c9",\n    "split.json": "cb9dd2e6702a97be13f18297a83f873c75ffacfe6dc8739c6bb20fe57c8491b8",\n    "unet/context.json": "cb5ec2e8b385de56e7aca519cacf981b05658372f114882d1a48eb882781c3f7",\n    "unet/network.pt": "873ce09e5f28c4df63e712e295cbcb18b53c60832e598d40602c204b880aafb0",\n    "unet/rainfall_climatology.nc": "56ad2218705c6dcbe1ccf6418a326a2076fc1b4296d0520804357e6e5544c2cb",\n    "unet/references.npz": "b4cb6eeb9e6b42261c207fabc42e4e09a196fab75f4bfd8aa954a5ad3e1bac40",\n    "unet/scaling.npz": "a969504274d065b32e9187dcfe01ed7027381f1d16432899bb543eb540736e18",\n    "unet/training.json": "c3484ceb692f763fcb8f3a7234a4997e7793722ee8d79440749df565a37ce8d2",\n    "unet/training_history.json": "c5ffde6f6a9e714b0139ec6994096977ae5bc1decfd39ea287e075bba8844bd7"\n  },\n  "forecast_issued": false,\n  "frozen_at": "2026-09-30T02:44:48.421423+00:00",\n  "independent_holdout": false,\n  "model_promoted": false,\n  "observation_file_sha256": "012bebbe0e38aefdf4f9e2432c3ca22f71d94ea1a1b5421da9d9e5048632cab0",\n  "schema": "rainfall_temporal_extension_frozen_v1",\n  "selected_epochs": 8,\n  "started_at": "2026-09-30T02:34:05.287588+00:00"\n}\n'}
BUNDLE_SHA256 = {'research/common/inputs.py': 'd22140d4ad3db4ac3fecdca26c7ebdd1d780486b51622a1f333fce2a637e6851', 'research/diagnostics/analyze_errors.py': '21ee50ab5d06fbb8ae59a524553ecac4121bb40136b77ac9b99299eff53f9067', 'research/fixed_blend/evaluate.py': '6c90f6cb96eaa2841fcec4ad57921b3bf9382f8143d3d2a4b8b04996e1cb44a9', 'research/sst_extension/evidence/completed_20260930/evaluation/global.csv': '6efdee5695689093202c639e46f58e5b2254db3454a5ac7eed48ef42b6f9ab3f', 'research/sst_extension/evidence/completed_20260930/frozen.json': '84c7c54e66686bbaa22f4aa15d50e2f9d19a5ab95a7637bde501eb06c8b46d23', 'research/sst_unet_blend/__init__.py': 'bf61b3fab3d44f939bbb448de4d41d7174519520b7938680f8e47e3a38e66c35', 'research/sst_unet_blend/experiment.py': '3e60d4e4199a4af1298025d280cbf25efe8deead78707a12ed53e8ff08a0b388', 'research/sst_unet_blend/protocol.json': '57312ad188255b70b9b3103ef26d63d1c911971ff9f480c2c0c522f7aaffc7ae', 'research/sst_unet_blend/test_blend.py': '08d50a3dc39bdd74562b386b1d5417c77f1165cceb3f08c748414e93a5d98241', 'research/temporal_extension/evidence/kaggle_354026629/evaluation/global.csv': '044c310fe49c905b01b4963704cc550c31da6a9ac0bdaaf531361e569a378f36', 'research/temporal_extension/evidence/kaggle_354026629/frozen.json': '915ed044d66fd24406d9d688505b718f02da8f2b92c7fefeb94bbc03c8eaaa60'}
for name, content in BUNDLE.items():
    assert hashlib.sha256(content.encode('utf-8')).hexdigest() == BUNDLE_SHA256[name]
    path = CODE_ROOT/name
    if path.exists() and path.read_text(encoding='utf-8') != content:
        raise RuntimeError('Different source code already exists. Preserve this run and choose a new RUN_ROOT.')
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8', newline='\n')
sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT/'research/prospective'))
print('Reviewed source bundle:', CODE_ROOT)


## 1. Synthetic checks
No neural or tree model is fitted, including in these checks.

In [ ]:
# A reused Kaggle kernel may still hold research modules from another notebook.
import importlib, sys
from pathlib import Path
CODE_ROOT = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
             else Path.cwd()/'outputs')/'sst_unet_fixed_blend'/'code'
for relative in [
    'research/sst_extension/evidence/completed_20260930/frozen.json',
    'research/temporal_extension/evidence/kaggle_354026629/frozen.json',
]:
    assert (CODE_ROOT/relative).is_file(), 'Run the source-bundle cell first: ' + relative
for name in list(sys.modules):
    if name == 'research' or name.startswith('research.'):
        del sys.modules[name]
sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()
from research.common.inputs import ROOT as ACTIVE_CODE_ROOT
assert ACTIVE_CODE_ROOT.resolve() == CODE_ROOT.resolve(), 'Unexpected source directory.'
print('Active comparison code:', ACTIVE_CODE_ROOT)
import unittest
suite = unittest.defaultTestLoader.loadTestsFromName("research.sst_unet_blend.test_blend")
assert unittest.TextTestRunner(verbosity=2).run(suite).wasSuccessful(), "Synthetic checks failed."


## 2. Locate the exact saved maps
Attach the original observations, the complete SST output and the complete U-Net extension output. Reports ZIPs are insufficient. Keep `None` for automatic discovery, or supply complete file paths.

In [ ]:
from research.sst_unet_blend.experiment import protocol, locate_files, execute, export_reports
OBSERVATIONS = None   # Full path to treino_tp.nc, if needed
SST_PREDICTIONS = None   # Full path to the SST models/predictions.nc, if needed
UNET_PREDICTIONS = None  # Full path to the U-Net extension predictions.nc, if needed
INPUTS = locate_files(protocol(), dict(observations=OBSERVATIONS,
    sst_predictions=SST_PREDICTIONS, unet_predictions=UNET_PREDICTIONS))
print("All required maps are present. No fitting will be performed.")


## 3. Evaluate the fixed 75/25 candidate
One previously consulted block, 2021–2022. Primary comparison: the existing hybrid/U-Net blend. Secondary comparison: the SST hybrid alone.

In [ ]:
import pandas as pd
from IPython.display import display, HTML, FileLink
OUTPUT = RUN_ROOT / "results"
RESULT = execute(OUTPUT, **INPUTS)
assert RESULT["complete_comparison"] and RESULT["archived_scores_reproduced"]
for name in ["global_metrics", "years", "regions"]:
    print(name)
    display(pd.read_csv(OUTPUT / (name + ".csv")))
display(HTML((OUTPUT / "report.html").read_text(encoding="utf-8")))
REPORT = export_reports(OUTPUT)
display(FileLink(str(REPORT.relative_to(BASE))))
print("Complete comparison:", RESULT["complete_comparison"])
print("Models fitted:", RESULT["model_fits"])
print("Download sst_unet_blend_reports.zip and preserve the complete output.")
